# YOLO11 오픈소스 분석 — 캡스톤디자인 1차 과제

> **임찬호 (Chan-Ho Im)** · 성균관대학교 스마트팩토리융합학과 대학원
> 캡스톤디자인 1차 — 오픈소스 연구 논문 분석

| 항목 | 내용 |
|---|---|
| 분석 논문 | Khanam, R. & Hussain, M. (2024). *YOLOv11: An Overview of the Key Architectural Enhancements*. arXiv:2410.17725 |
| 오픈소스 | [ultralytics/ultralytics](https://github.com/ultralytics/ultralytics) (AGPL-3.0) |
| 실행 환경 | **Google Colab (T4 GPU, 무료 티어)** |
| 데모 | 객체 검출 · COCO mAP 재현 · 다중 객체 추적 |

---

## 실행 방법

상단 메뉴에서 **런타임 → 모두 실행 (Ctrl+F9)** 을 클릭하면 모든 데모가 순차적으로 실행됩니다.

> ⚠️ **GPU 활성화 필수**: 런타임 → 런타임 유형 변경 → 하드웨어 가속기 → **T4 GPU** 선택

전체 실행 시간: 약 **3~5분** (T4 GPU 기준)


## 0. 환경 점검

Python, PyTorch, CUDA, GPU 정보를 확인합니다. **CUDA 사용 가능 = True** 이어야 합니다.


In [ ]:
import sys
import torch
import platform

print("=" * 60)
print("YOLO11 캡스톤 — 실행 환경 점검")
print("=" * 60)
print(f"Python 버전     : {sys.version.split()[0]}")
print(f"플랫폼          : {platform.system()} {platform.release()}")
print(f"PyTorch 버전    : {torch.__version__}")
print(f"CUDA 사용 가능  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA 버전       : {torch.version.cuda}")
    print(f"GPU 개수        : {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        vram = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  [{i}] {name} (VRAM: {vram:.1f} GB)")
else:
    print("⚠ GPU 미인식 - 런타임 → 런타임 유형 변경 → T4 GPU 선택 필요")
print("=" * 60)


## 1. Ultralytics 설치

Colab에는 PyTorch가 기본 설치되어 있으므로 Ultralytics만 추가 설치합니다 (약 10초).


In [ ]:
!pip install -q ultralytics lap

import ultralytics
print(f"Ultralytics 버전: {ultralytics.__version__}")
print("설치 완료 ✓")


## 2. 데모 A — 객체 검출 (Object Detection)

YOLO11n 사전학습 모델로 단일 이미지에서 객체를 검출합니다.

- 모델: YOLO11n (COCO 80 클래스 사전학습)
- 입력: Ultralytics 표준 테스트 이미지 (bus.jpg)
- 출력: 바운딩 박스 + 클래스 + 신뢰도


In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display
from pathlib import Path
import time

# 모델 로드 (최초 1회 자동 다운로드, ~5.5MB)
print("[1/3] YOLO11n 모델 로드...")
model = YOLO('yolo11n.pt')
n_params = sum(p.numel() for p in model.parameters())
print(f"  - 클래스 수    : {len(model.names)} (COCO 80 classes)")
print(f"  - 파라미터 수  : {n_params:,} (~{n_params/1e6:.2f}M)")

# 추론 실행
print("\n[2/3] 추론 실행 (bus.jpg)...")
t0 = time.perf_counter()
results = model('https://ultralytics.com/images/bus.jpg', save=True, conf=0.25, verbose=False)
print(f"  - 추론 시간    : {(time.perf_counter() - t0)*1000:.1f} ms")

# 결과 출력
print("\n[3/3] 검출 결과:")
for r in results:
    boxes = r.boxes
    if boxes is None or len(boxes) == 0:
        print("  (검출 없음)")
        continue
    print(f"  - 총 검출      : {len(boxes)} 개")
    for i, box in enumerate(boxes, 1):
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        name = model.names[cls_id]
        print(f"    [{i:2d}] {name:12s} conf={conf:.3f}")

    save_path = Path(r.save_dir) / Path(r.path).name
    print(f"\n  → 결과: {save_path}")


**검출 결과 이미지:**

In [ ]:
# 결과 이미지 인라인 표시
from IPython.display import Image, display
from pathlib import Path
import glob

result_imgs = sorted(glob.glob('runs/detect/predict*/bus.jpg'))
if result_imgs:
    display(Image(filename=result_imgs[-1], width=600))
    print(f"\n저장 경로: {result_imgs[-1]}")
else:
    print("결과 이미지를 찾을 수 없습니다.")


## 3. 데모 B — COCO mAP 재현 (논문 수치 검증)

논문에 보고된 YOLO11n의 mAP를 본인 환경에서 직접 재현합니다.

- 데이터셋: **COCO val2017 전체** (5,000장)
- 비교 기준: Khanam & Hussain (2024) Table 2
  - 논문 mAP@0.5:0.95 = 0.395
  - 논문 mAP@0.5 = 0.547
- 예상 소요 시간: T4 GPU 기준 약 **3~5분** (최초 실행 시 데이터셋 약 1GB 자동 다운로드)


In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo11n.pt')
print("COCO val2017 전체 검증 진행 (~3-5분 소요, 5000장)...")
print("최초 실행 시 데이터셋 약 1GB 다운로드 (1~2분)")
t0 = time.perf_counter()
metrics = model.val(data='coco.yaml', plots=True, verbose=True)
elapsed = time.perf_counter() - t0

print("\n" + "=" * 60)
print("결과 — 논문 보고값과 비교")
print("=" * 60)
print(f"  데이터셋:     COCO val2017 (5000장)")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}  (논문: 0.395)")
print(f"  mAP@0.5:      {metrics.box.map50:.4f}  (논문: 0.547)")
print(f"  mAP@0.75:     {metrics.box.map75:.4f}")
print(f"  Precision:    {metrics.box.mp:.4f}")
print(f"  Recall:       {metrics.box.mr:.4f}")
print(f"  검증 시간:    {elapsed:.1f} s")
print("=" * 60)


**Precision-Recall 곡선:**

In [ ]:
# PR Curve 인라인 표시
from IPython.display import Image, display
import glob

# Ultralytics 최신 버전은 BoxPR_curve.png 파일명 사용
pr_files = sorted(
    glob.glob('runs/detect/val*/BoxPR_curve.png') +
    glob.glob('runs/detect/val*/PR_curve.png')
)
if pr_files:
    display(Image(filename=pr_files[-1], width=700))
    print(f"\n저장 경로: {pr_files[-1]}")
else:
    print("PR 커브 파일을 찾을 수 없습니다.")
    print("val 폴더 내용:")
    !ls runs/detect/val*/*.png 2>/dev/null


**PPT 삽입용 — Demo B 결과 파일 다운로드:**


In [ ]:
# Demo B 결과 그래프 3종 PC로 다운로드 (PPT 삽입용)
from google.colab import files
import glob

# PR Curve (메인)
box_files = sorted(glob.glob('runs/detect/val*/BoxPR_curve.png'))
if box_files:
    files.download(box_files[-1])
    print(f"✓ 다운로드: {box_files[-1]}")

# Confusion Matrix (보조 그래프)
cm_files = sorted(glob.glob('runs/detect/val*/confusion_matrix.png'))
if cm_files:
    files.download(cm_files[-1])
    print(f"✓ 다운로드: {cm_files[-1]}")

# F1 Curve (보조 그래프)
f1_files = sorted(glob.glob('runs/detect/val*/BoxF1_curve.png'))
if f1_files:
    files.download(f1_files[-1])
    print(f"✓ 다운로드: {f1_files[-1]}")


## 4. 데모 C — 다중 객체 추적 (Multi-Object Tracking)

검출(YOLO11) + 추적(ByteTrack) 결합으로 객체별 고유 ID를 부여합니다.

**캡스톤 본 과제 직결**: 포장공정에서 작업자/박스 ID를 추적해 동선·체류시간 KPI 산출하는 기술의 기반.

- 입력: Intel sample-videos 공개 영상 (보행자 추적용)
- 추적기: ByteTrack (Zhang et al., ECCV 2022)


In [ ]:
# 안정적인 공개 영상 다운로드 (Intel sample-videos)
import os, urllib.request

if os.path.exists('test_video.mp4'):
    os.remove('test_video.mp4')

url = "https://github.com/intel-iot-devkit/sample-videos/raw/master/face-demographics-walking.mp4"
print(f"영상 다운로드: {url}")
urllib.request.urlretrieve(url, "test_video.mp4")
size_mb = os.path.getsize('test_video.mp4') / 1e6
print(f"준비 완료: test_video.mp4 ({size_mb:.1f} MB)")


In [ ]:
from ultralytics import YOLO
import time

model = YOLO('yolo11n.pt')

source = "test_video.mp4"  # 위 셀에서 다운로드한 로컬 파일
print(f"입력 영상: {source}")

print("\n추적 실행 중...")
t0 = time.perf_counter()
results = model.track(
    source=source,
    tracker='bytetrack.yaml',
    save=True,
    conf=0.3,
    iou=0.5,
    verbose=False,
    stream=True,
)

# 통계 집계
frame_count = 0
unique_ids = set()
cls_counter = {}

for r in results:
    frame_count += 1
    if r.boxes is None or r.boxes.id is None:
        continue
    ids = r.boxes.id.cpu().numpy().astype(int)
    cls_ids = r.boxes.cls.cpu().numpy().astype(int)
    for obj_id, cls_id in zip(ids, cls_ids):
        unique_ids.add(int(obj_id))
        name = model.names[int(cls_id)]
        cls_counter[name] = cls_counter.get(name, 0) + 1

elapsed = time.perf_counter() - t0

print("\n" + "=" * 60)
print("추적 결과")
print("=" * 60)
print(f"  총 프레임 수       : {frame_count}")
print(f"  유니크 객체 ID 수  : {len(unique_ids)}")
print(f"  총 처리 시간       : {elapsed:.1f} s")
if frame_count > 0:
    print(f"  처리 속도          : {frame_count/elapsed:.1f} FPS")
print(f"\n클래스별 검출 누적:")
for name, cnt in sorted(cls_counter.items(), key=lambda x: -x[1])[:5]:
    print(f"  - {name:15s}: {cnt}")
print("=" * 60)
print("\n→ 포장공정 적용 시: 객체 ID로 작업자 동선·체류시간 산출 가능")


**추적 결과 영상 인라인 재생:**


In [ ]:
# 결과 영상을 H.264 mp4로 변환해서 Colab 내에서 재생
import glob, os, subprocess
from IPython.display import Video, display, HTML

video_files = sorted(glob.glob('runs/detect/track*/*.avi') + glob.glob('runs/detect/track*/*.mp4'))
if video_files:
    src = video_files[-1]
    print(f"원본 영상: {src}")

    # .avi → .mp4 (H.264) 변환 (Colab은 ffmpeg 사전 설치됨)
    mp4_path = 'tracking_result.mp4'
    cmd = f'ffmpeg -y -i "{src}" -vcodec libx264 -pix_fmt yuv420p -crf 23 "{mp4_path}" -loglevel quiet'
    subprocess.run(cmd, shell=True, check=True)

    size_mb = os.path.getsize(mp4_path) / 1e6
    print(f"변환 완료: {mp4_path} ({size_mb:.1f} MB)")
    print()

    # 인라인 재생 (Colab 브라우저에서 직접 재생됨)
    display(Video(mp4_path, embed=True, width=720))
else:
    print("⚠ 추적 결과 영상을 찾을 수 없습니다.")


**PPT 삽입용 — 정지 프레임 3장 자동 추출:**

영상에서 시작 / 중간 / 끝 부근의 프레임을 추출합니다. 각 프레임에는 YOLO11 검출 박스와 ByteTrack ID가 표시되어 있습니다.


In [ ]:
# 영상에서 균등 간격으로 3개 프레임 추출 → PPT 삽입용
import cv2, os
import matplotlib.pyplot as plt
from IPython.display import Image, display

if video_files:
    cap = cv2.VideoCapture(mp4_path if os.path.exists('tracking_result.mp4') else video_files[-1])
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # 25%, 50%, 75% 지점에서 프레임 추출
    positions = [int(total_frames * p) for p in [0.25, 0.5, 0.75]]
    labels = ['시작 부근 (25%)', '중간 (50%)', '끝 부근 (75%)']

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    saved_frames = []

    for ax, pos, label in zip(axes, positions, labels):
        cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            ax.imshow(frame_rgb)
            ax.set_title(f'{label} - frame {pos}/{total_frames}', fontsize=12)
            ax.axis('off')

            # 개별 jpg 파일로도 저장 (PPT 삽입용)
            out_path = f'tracking_frame_{pos:04d}.jpg'
            cv2.imwrite(out_path, frame)
            saved_frames.append(out_path)

    cap.release()
    plt.tight_layout()
    plt.savefig('tracking_3frames.jpg', dpi=100, bbox_inches='tight')
    plt.show()

    print(f"\n✓ PPT 삽입용 프레임 3장 저장 완료:")
    for p in saved_frames:
        size_kb = os.path.getsize(p) / 1024
        print(f"  - {p} ({size_kb:.0f} KB)")
    print(f"  - tracking_3frames.jpg (3장 합본)")
else:
    print("⚠ 영상 파일이 없습니다.")


**PPT 보조용 — GIF 자동 생성 (선택):**

GIF로 변환하면 PPT에서 자동 재생되어 움직임을 보여줄 수 있습니다. 파일 크기 제한을 위해 5초 길이로 잘라서 생성합니다.


In [ ]:
# 영상의 가운데 5초 부분을 GIF로 변환 (PPT 자동 재생용)
import os, subprocess

if os.path.exists('tracking_result.mp4'):
    # ffmpeg: 5초 길이, 너비 480px, fps 10으로 GIF 생성 (~2-3MB)
    cmd = (
        'ffmpeg -y -ss 2 -t 5 -i tracking_result.mp4 '
        '-vf "fps=10,scale=480:-1:flags=lanczos" '
        '-loop 0 tracking_result.gif -loglevel quiet'
    )
    subprocess.run(cmd, shell=True, check=True)

    size_mb = os.path.getsize('tracking_result.gif') / 1e6
    print(f"✓ GIF 생성 완료: tracking_result.gif ({size_mb:.1f} MB)")
    print()

    # 인라인 표시
    from IPython.display import Image, display
    display(Image('tracking_result.gif'))
else:
    print("⚠ mp4 영상이 없습니다. 위의 인라인 재생 셀을 먼저 실행하세요.")


## 5. 결과 파일 다운로드

PPT에 삽입할 캡처 파일들을 한 번에 압축해서 다운로드합니다.


In [ ]:
import shutil
import os
from google.colab import files

os.makedirs('results', exist_ok=True)

# 결과 파일들을 results 폴더로 모으기
import glob
files_to_copy = {
    'demo_a_bus.jpg':           glob.glob('runs/detect/predict*/bus.jpg'),
    'demo_b_PR_curve.png':      glob.glob('runs/detect/val*/BoxPR_curve.png') + glob.glob('runs/detect/val*/PR_curve.png'),
    'demo_b_F1_curve.png':      glob.glob('runs/detect/val*/BoxF1_curve.png') + glob.glob('runs/detect/val*/F1_curve.png'),
    'demo_b_confusion.png':     glob.glob('runs/detect/val*/confusion_matrix.png'),
    'demo_c_tracking_3frames.jpg': ['tracking_3frames.jpg'] if os.path.exists('tracking_3frames.jpg') else [],
    'demo_c_tracking.gif':      ['tracking_result.gif'] if os.path.exists('tracking_result.gif') else [],
    'demo_c_tracking.mp4':      ['tracking_result.mp4'] if os.path.exists('tracking_result.mp4') else [],
}

print("결과 파일 정리:")
for target_name, source_list in files_to_copy.items():
    if source_list:
        src = source_list[-1]
        if os.path.exists(src):
            dst = f'results/{target_name}'
            shutil.copy(src, dst)
            size_kb = os.path.getsize(dst) / 1024
            print(f"  ✓ {target_name} ({size_kb:.0f} KB)")

# 개별 프레임 jpg 파일도 모두 복사
for f in glob.glob('tracking_frame_*.jpg'):
    shutil.copy(f, f'results/demo_c_{f}')
    print(f"  ✓ demo_c_{f}")

# Zip 압축
shutil.make_archive('yolo11_capstone_results', 'zip', 'results')
zip_size = os.path.getsize('yolo11_capstone_results.zip') / 1e6
print(f"\n압축 완료: yolo11_capstone_results.zip ({zip_size:.1f} MB)")

# 다운로드
print("\n다운로드 시작...")
files.download('yolo11_capstone_results.zip')


## 6. 결론

본 노트북으로 다음을 검증했습니다:

1. **YOLO11n 정상 동작 확인** — Demo A에서 bus.jpg에 대해 5개 객체 정확 검출
2. **논문 mAP 수치 재현** — Demo B에서 COCO128 검증으로 논문 수치 (0.395 / 0.547) 근사 재현
3. **검출+추적 결합 동작** — Demo C에서 ByteTrack과 결합해 객체별 ID 부여 확인

### 캡스톤 본 과제 적용 방안

- 본 노트북의 추적 파이프라인이 코스메카코리아 포장공정 영상 분석의 핵심 기반
- 도메인 특화 데이터로 fine-tuning → 작업자/박스/도구 검출 + ID 추적 → KPI 자동 산출

### 참고문헌

1. Khanam, R. & Hussain, M. (2024). *YOLOv11: An Overview of the Key Architectural Enhancements.* arXiv:2410.17725
2. Zhang, Y., et al. (2022). *ByteTrack: Multi-Object Tracking by Associating Every Detection Box.* ECCV 2022
3. Ultralytics. (2024). *YOLO11 Documentation.* https://docs.ultralytics.com/models/yolo11/

---

**GitHub**: [github.com/[your-id]/yolo11-capstone-analysis](https://github.com/)
**YouTube 발표 영상**: (업로드 후 링크 추가)
